# Índice Figurinha da Copa

Quanto custa fechar o álbum sozinho, sem troca, dividido pelo salário mínimo.

Premissas:

- álbum com 980 cromos;
- 7 cromos por pacote;
- todo cromo tem a mesma chance;
- não vem cromo repetido no mesmo pacote;
- repetida entre pacotes vai para o monte;
- salário de 14 pagamentos vira base anual / 12.

## 0. Setup

In [ ]:
import csv
import random
import statistics
from math import ceil, log
from pathlib import Path

PROJECT_DIR = Path.cwd()
DATA_DIR = PROJECT_DIR / "data"
OUTPUT_CSV = DATA_DIR / "figurinha_index.csv"

TOTAL_STICKERS = 980
PER_PACK = 7
PACK_PRICE_BR = 7
TRIALS = 30_000
SEED = 2026

# Sorte perfeita: nenhum repetido. Uso ceil para não depender de divisão exata.
PACKS_PERFECT = ceil(TOTAL_STICKERS / PER_PACK)

print(f"Cromos na colecao: {TOTAL_STICKERS}")
print(f"Cromos por pacote: {PER_PACK}")
print(f"Pacotes com sorte perfeita: {PACKS_PERFECT}")

## 1. Datasets Readings

In [ ]:
# Tudo em moeda local.
# usd = quanto vale 1 unidade local em dólar.
# min_wage está como a fonte publica. Espanha e Portugal pagam 14 vezes ao ano;
# ajusto para base 12 na conta do índice.

countries = [
    {"pais": "Brasil", "moeda": "R$", "pack": 7.00, "album": 24.90, "min_wage": 1621.00, "usd": 1 / 5.1855, "pay14": False},
    {"pais": "EUA (fed.)", "moeda": "US$", "pack": 2.00, "album": 5.00, "min_wage": 1256.50, "usd": 1.0, "pay14": False},
    {"pais": "Canada", "moeda": "C$", "pack": 2.70, "album": 22.00, "min_wage": 3153.00, "usd": 1 / 1.394, "pay14": False},
    {"pais": "Mexico", "moeda": "MX$", "pack": 25.00, "album": 99.00, "min_wage": 9582.47, "usd": 1 / 17.386, "pay14": False},
    {"pais": "Argentina", "moeda": "AR$", "pack": 2000.00, "album": 15000.00, "min_wage": 367800.00, "usd": 1 / 1460.0, "pay14": False},
    {"pais": "Reino Unido", "moeda": "£", "pack": 1.25, "album": 4.99, "min_wage": 2065.00, "usd": 1.3377, "pay14": False},
    {"pais": "Espanha", "moeda": "€", "pack": 1.50, "album": 5.00, "min_wage": 1221.00, "usd": 1.154, "pay14": True},
    {"pais": "Alemanha", "moeda": "€", "pack": 1.50, "album": 5.00, "min_wage": 2343.00, "usd": 1.154, "pay14": False},
    {"pais": "Franca", "moeda": "€", "pack": 1.50, "album": 5.00, "min_wage": 1867.02, "usd": 1.154, "pay14": False},
    {"pais": "Portugal", "moeda": "€", "pack": 1.50, "album": 5.00, "min_wage": 920.00, "usd": 1.154, "pay14": True},
]

len(countries), countries[0]

## 2. Transformations

### 2.1 Coupon Collector baseline

In [ ]:
# Fórmula clássica: uma figurinha por sorteio, com reposição.
# Serve como referência para a simulação.

harmonic = sum(1 / i for i in range(1, TOTAL_STICKERS + 1))
expected_stickers = TOTAL_STICKERS * harmonic
expected_packs_analytic = expected_stickers / PER_PACK

print(f"H_{TOTAL_STICKERS}: {harmonic:.6f}")
print(f"Figurinhas esperadas no modelo classico: {expected_stickers:,.1f}")
print(f"Pacotes esperados no modelo classico: {expected_packs_analytic:,.1f}")

### 2.2 Monte Carlo simulation

In [ ]:
# Aqui eu imito abrir pacote de verdade.
# Cada álbum começa vazio.
# Cada pacote traz 7 cromos diferentes.
# Repetida entre pacotes não cola.
# No fim, guardo quantos pacotes foram.
#
# Quer mudar o tamanho do álbum? Mude TOTAL_STICKERS no Setup.
# Isso muda os cromos possíveis, a meta do while e o custo final.

rng = random.Random(SEED)
ids = list(range(TOTAL_STICKERS))
simulation_results = []

for trial in range(TRIALS):
    owned = set()
    packs = 0

    while len(owned) < TOTAL_STICKERS:
        pack = rng.sample(ids, PER_PACK)  # sem repetição no pacote
        owned.update(pack)
        packs += 1

    simulation_results.append(packs)

simulation_results.sort()

sim_summary = {
    "mean": statistics.mean(simulation_results),
    "median": statistics.median(simulation_results),
    "p10": simulation_results[int(0.10 * TRIALS)],
    "p90": simulation_results[int(0.90 * TRIALS)],
    "min": simulation_results[0],
    "max": simulation_results[-1],
    "trials": TRIALS,
}

sim_summary

### 2.3 Como a quantidade de cromos muda a conta

In [ ]:
# Conta rápida para o efeito do tamanho do álbum.
# Cresce mais ou menos como n * log(n) / cromos_por_pacote.

scenarios = [
    {"label": "Album menor, pacote 5", "total": 670, "per_pack": 5},
    {"label": "Album 2026", "total": 980, "per_pack": 7},
    {"label": "Album 2026, pacote 5", "total": 980, "per_pack": 5},
    {"label": "Album 1200, pacote 7", "total": 1200, "per_pack": 7},
]

for s in scenarios:
    h = sum(1 / i for i in range(1, s["total"] + 1))
    perfect = ceil(s["total"] / s["per_pack"])
    analytic = s["total"] * h / s["per_pack"]
    print(f'{s["label"]:<24} total={s["total"]:>4} per_pack={s["per_pack"]} '
          f'perfect={perfect:>4} expected={analytic:>7.1f}')

### 2.4 Country index

In [ ]:
packs_solo = round(sim_summary["mean"])
rows = []

for c in countries:
    wage_12 = c["min_wage"] * 14 / 12 if c["pay14"] else c["min_wage"]
    cost_min = c["album"] + PACKS_PERFECT * c["pack"]
    cost_solo = c["album"] + packs_solo * c["pack"]

    rows.append({
        "pais": c["pais"],
        "moeda": c["moeda"],
        "pacote_local": round(c["pack"], 2),
        "album_local": round(c["album"], 2),
        "custo_min_local": round(cost_min, 2),
        "custo_solo_local": round(cost_solo, 2),
        "custo_solo_usd": round(cost_solo * c["usd"], 2),
        "salario_min_mensal_local": round(wage_12, 2),
        "salario_min_usd": round(wage_12 * c["usd"], 2),
        "indice_min_pct": round(100 * cost_min / wage_12, 1),
        "indice_solo_pct": round(100 * cost_solo / wage_12, 1),
        "meses_de_salario": round(cost_solo / wage_12, 2),
    })

rows.sort(key=lambda row: row["indice_solo_pct"], reverse=True)
rows[:3]

### 2.5 Salvar CSV

In [ ]:
DATA_DIR.mkdir(parents=True, exist_ok=True)

with OUTPUT_CSV.open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
    writer.writeheader()
    writer.writerows(rows)

print(f"CSV salvo em: {OUTPUT_CSV}")

## 3. Verifications

In [ ]:
assert TOTAL_STICKERS == 980
assert PER_PACK == 7
assert PACKS_PERFECT == 140
assert round(expected_packs_analytic, 1) == 1045.1
assert sim_summary["p10"] == 848
assert sim_summary["p90"] == 1274
assert abs(sim_summary["mean"] - expected_packs_analytic) / expected_packs_analytic < 0.01
assert rows[0]["pais"] == "Argentina"
assert rows[1]["pais"] == "Brasil"
assert rows[0]["meses_de_salario"] == 5.71
assert rows[1]["meses_de_salario"] == 4.52

print("Verificacoes passaram.")
print(f'Media Monte Carlo: {sim_summary["mean"]:.1f} pacotes')
print(f'Usando no indice: {packs_solo} pacotes')

In [ ]:
for row in rows:
    print(
        f'{row["pais"]:<12} '
        f'{row["moeda"]} {row["custo_solo_local"]:>10,.0f} '
        f'{row["indice_solo_pct"]:>6.1f}% '
        f'{row["meses_de_salario"]:>4.2f} meses'
    )